In [ ]:
#!pip install pyomo
#!pip install highspy

In [1]:
# Import libraries
from pyomo.environ import Param, Set, ConcreteModel, Var, Constraint, NonNegativeReals, Objective, maximize, minimize, SolverFactory, Suffix

# Despacho económico

En el despacho nos interesa maximizar el benficio social, es decir, atender la demanda mientras mínimo los costos de producción

BS = Valor de la atención de la demanda - Costo de produccion

## Sets - conjuntos

$g \in G = \{g1, g2, g3\} : \text{Generadores}$  
$d \in D = \{d1, d2, d3, d4\} : \text{Demanda}$  
$l \in L = \{l1\} : \text{Lineas}$  
$s \in S = \{s1, s2\} : \text{barras}$  

## Parámetros

$P^{max}_{g}$: Potencia máxima del generador

Curvas de valoración y de costo  
$c^{dem}_{d}$: curva de cantidades y precio de demada  
$c^{gen}_{g}$: curva de cantidades y precios de la generación  
$Cap^{max}_{l}$: Capacidad máxima de la línea  

## Variablers

$L_{d}$: Demanda de la carga d  
$P_{g}$: Generación del generador g  
$I_{l}$: Flujo de la línea L de importanción  
$E_{l}$: Flujo de la línea L de exportación  

## Ecuaciones

### Función objetivo

$\max \: \sum_{d} L_{d} \cdot c^{dem}_{d} - \sum_{g} P_{g} \cdot c^{gen}_{g} $

### Restricciones
 
Balance de potencia:  
$\sum_{d \in s} L_{d} + \sum_{l \in s} E_{l}= \sum_{g \in s} P_{g} + \sum_{l \in s} I_{l} \; \forall s \; : \lambda_{s}$  
$\sum_{l \in s} E_{l} = - \sum_{l \in s} I_{l} \; \forall s \; : \eta$

# Límite máximo de los generadores  
$ P_{g} \le P^{max}_{g} \; \forall g$

# Límite máximo de demanda
$ L_{d} \le D^{max}_{d} \; \forall d$

# Creación del modelo y conjunts

In [2]:
# Definición del modelo
m = ConcreteModel("Despacho uninodal")

# Definición de los conjuntos
m.GENERATORS = Set(initialize=["Hydro", "Coal", "Gas"])
m.DEMAND_BLOCKS = Set(initialize=["D1", "D2", "D3", "D4"])
m.BUSES = Set(initialize=["Bus1", "Bus2"])
m.LINES = Set(initialize=["Line1"])

# Carga de datos (puede ser de un DB, excel, u otros)

In [3]:
generator_capacity = {
    "Hydro": 100,
    "Coal": 80,
    "Gas": 120,
}

generator_cost = {
    "Hydro": 20,
    "Coal": 40,
    "Gas": 90,
}

demand_capacity = {
    "D1": 50,
    "D2": 50,
    "D3": 50,
    "D4": 50,
}

demand_price = {
    "D1": 150,
    "D2": 110,
    "D3": 70,
    "D4": 30,
}

map_gen = {
    "Hydro": "Bus1",
    "Coal": "Bus1",
    "Gas": "Bus2",
}

map_demand = {
    "D1": "Bus1",
    "D2": "Bus1",
    "D3": "Bus1",
    "D4": "Bus2",
}

line_cap = {
    "Line1": 20,
}

In [4]:
# pasamos los parámetros al objeto de nuestro módelo, para poder llamarlo de manera simplificada
m.GenMax = Param(
    m.GENERATORS,
    initialize=generator_capacity
)

m.GenCost = Param(
    m.GENERATORS,
    initialize=generator_cost
)

m.DemandMax = Param(
    m.DEMAND_BLOCKS,
    initialize=demand_capacity
)

m.DemandPrice = Param(
    m.DEMAND_BLOCKS,
    initialize=demand_price
)

m.MapGen = Param(
    m.GENERATORS,
    initialize=map_gen
)

m.MapDemand = Param(
    m.DEMAND_BLOCKS,
    initialize=map_demand
)

m.line_cap = Param(
    m.LINES,
    initialize=line_cap
)

The default domain for Param objects is 'Any'.  However, we will be changing
that default to 'Reals' in the future.  If you really intend the domain of
this Paramto be 'Any', you can suppress this warning by explicitly specifying
'within=Any' to the Param constructor. (deprecated in 5.6.9, will be removed
in (or after) 6.0) (called from D:\ambiente\datos de
programa\Python\Python311\site-
packages\pyomo\core\base\indexed_component.py:756)
'Any'. The default domain for Param objects is 'Any'.  However, we will be
changing that default to 'Reals' in the future.  If you really intend the
domain of this Paramto be 'Any', you can suppress this warning by explicitly
specifying 'within=Any' to the Param constructor.  (deprecated in 5.6.9, will
be removed in (or after) 6.0) (called from D:\ambiente\datos de
programa\Python\Python311\site-
packages\pyomo\core\base\indexed_component.py:756)


# Variables

In [5]:
# Generation dispatch
m.pg = Var(m.GENERATORS, domain=NonNegativeReals)

# Accepted demand
m.pd = Var(m.DEMAND_BLOCKS, domain=NonNegativeReals)

m.pf = Var(m.LINES, domain=NonNegativeReals)

# Modelo
## Función objetivo

In [6]:
def social_wellfare(model):
  #valor_demanda = sum([model.pd[d] * model.DemandPrice[d] for d in model.DEMAND_BLOCKS])
  costo_gen = sum([model.pg[g] * model.GenCost[g] for g in model.GENERATORS])
  # return valor_demanda - costo_gen
  return costo_gen
# m.fo = Objective(rule=social_wellfare, sense=maximize)
m.fo = Objective(rule=social_wellfare, sense=minimize)

## Sujeto a:

In [23]:
# Balance de potencia
def balance(model, s):
  return (
    sum([model.pg[g] for g in model.GENERATORS if model.MapGen[g] == s]) 
    - (1 if s == "Bus1" else -1) * model.pf["Line1"] == 

    sum([model.pd[d] for d in model.DEMAND_BLOCKS if model.MapDemand[d] == s])
  )
m.balance = Constraint(m.BUSES, rule=balance, name='conservacion')

def max_flow(model, l):
  return model.pf[l] <= model.line_cap[l]
m.max_flow = Constraint(m.LINES, rule=max_flow, name='max_flow')

def fix_flow(model, l):
  return model.pf[l] >= 10
m.fix_flow = Constraint(m.LINES, rule=fix_flow, name='fix_flow')

def min_flow(model, l):
  return model.pf[l] >= - model.line_cap[l]
m.min_flow = Constraint(m.LINES, rule=min_flow, name='min_flow')

# Potencia máxima de los generadores
def max_gen(model, g):
  return model.pg[g] <= model.GenMax[g]
m.max_gen = Constraint(m.GENERATORS, rule=max_gen, name='max_gen')

# Conusmo máximo de los bloques de demanda
def max_demand(model, d):
  return model.pd[d] == model.DemandMax[d]
m.max_demand = Constraint(m.DEMAND_BLOCKS, rule=max_demand, name='max_demand')

'pyomo.core.base.constraint.IndexedConstraint'>) on block Despacho uninodal
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
'pyomo.core.base.constraint.IndexedConstraint'>) on block Despacho uninodal
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
'pyomo.core.base.constraint.IndexedConstraint'>) on block Despacho uninodal
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
'pyomo.core.base.constraint.IndexedConstraint'>) on block Despacho uninodal
with a new Component (type=<class
'pyomo.core.base.constraint.Indexed

# Solver

In [24]:
m.dual = Suffix(direction=Suffix.IMPORT)
solver = SolverFactory("highs")
options = {"symbolic_solver_labels" : True}
m.write('nodales.lp', io_options = options)
results = solver.solve(m, tee=True)

'pyomo.core.base.suffix.Suffix'>) on block Despacho uninodal with a new
Component (type=<class 'pyomo.core.base.suffix.Suffix'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP has 12 rows; 8 cols; 19 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+01, 9e+01]
  Bound   [0e+00, 0e+00]
  RHS     [1e+01, 1e+02]
Presolving model
1 rows, 3 cols, 3 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-12); columns 0(-8); nonzeros 0(-19) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  7.5000000000e+03
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [26]:
m.pprint()

4 Set Declarations
    BUSES : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'Bus1', 'Bus2'}
    DEMAND_BLOCKS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'D1', 'D2', 'D3', 'D4'}
    GENERATORS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'Hydro', 'Coal', 'Gas'}
    LINES : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    1 : {'Line1',}

8 Param Declarations
    DemandMax : Size=4, Index=DEMAND_BLOCKS, Domain=Any, Default=None, Mutable=False
        Key : Value
         D1 :    50
         D2 :    50
         D3 :    50
         D4 :    50
    DemandPrice : Size=4, Index=DEMAND_BLOCKS, Domain=Any, Default=None, Mutable=False
        Key : Value
         D1 :   150
         D2 

In [18]:
print(results)


Problem: 
- Lower bound: 9000.0
  Upper bound: 9000.0
  Number of objectives: 1
  Number of constraints: nan
  Number of variables: nan
  Sense: minimize
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.convergenceCriteriaSatisfied



In [19]:
m.pd.pprint()

pd : Size=4, Index=DEMAND_BLOCKS
    Key : Lower : Value : Upper : Fixed : Stale : Domain
     D1 :     0 :  50.0 :  None : False : False : NonNegativeReals
     D2 :     0 :  50.0 :  None : False : False : NonNegativeReals
     D3 :     0 :  50.0 :  None : False : False : NonNegativeReals
     D4 :     0 :  50.0 :  None : False : False : NonNegativeReals


In [25]:
m.pg.pprint()

pg : Size=3, Index=GENERATORS
    Key   : Lower : Value : Upper : Fixed : Stale : Domain
     Coal :     0 :  70.0 :  None : False : False : NonNegativeReals
      Gas :     0 :  30.0 :  None : False : False : NonNegativeReals
    Hydro :     0 : 100.0 :  None : False : False : NonNegativeReals


In [26]:
m.pf.pprint()

pf : Size=1, Index=LINES
    Key   : Lower : Value : Upper : Fixed : Stale : Domain
    Line1 :     0 :  20.0 :  None : False : False : NonNegativeReals


In [27]:
mcp1 = m.dual[m.balance["Bus1"]]
mcp2 = m.dual[m.balance["Bus2"]]

print(mcp1)
print(mcp2)

40.0
90.0


In [15]:
m.balance.pprint()

balance : Size=2, Index=BUSES, Active=True
    Key  : Lower : Body                                                          : Upper : Active
    Bus1 :   0.0 : pg[Hydro] + pg[Coal] + pf[Line1] - (pd[D1] + pd[D2] + pd[D3]) :   0.0 :   True
    Bus2 :   0.0 :                                  pg[Gas] - pf[Line1] - pd[D4] :   0.0 :   True
